In [2]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
!pip install shimmy

In [27]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.buffers import ReplayBuffer

In [45]:
# Create directories for logs and models
log_dir = "logs/"
model_dir = "models/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

In [46]:
"""def load_data(data_path):
    """"""
    Load and display stock data.

    Args:
        data_path (str): Path to the Excel/CSV file

    Returns:
        DataFrame: Loaded data or None if error
    """"""
    try:
        if data_path.endswith('.xlsx') or data_path.endswith('.xls'):
            data = pd.read_excel(data_path)
        else:  # Assume CSV
            data = pd.read_csv(data_path)

        print(f"Successfully loaded data from {data_path}")
        print(f"Data shape: {data.shape}")
        print(f"Column names: {data.columns.tolist()}")

        # Display sample of the data
        print(data.head())

        return data
    except Exception as e:
        print(f"Failed to load data: {e}")
        return None"""

'def load_data(data_path):\n    \n    Load and display stock data.\n    \n    Args:\n        data_path (str): Path to the Excel/CSV file\n        \n    Returns:\n        DataFrame: Loaded data or None if error\n    \n    try:\n        if data_path.endswith(\'.xlsx\') or data_path.endswith(\'.xls\'):\n            data = pd.read_excel(data_path)\n        else:  # Assume CSV\n            data = pd.read_csv(data_path)\n            \n        print(f"Successfully loaded data from {data_path}")\n        print(f"Data shape: {data.shape}")\n        print(f"Column names: {data.columns.tolist()}")\n\n        # Display sample of the data\n        print(data.head())\n\n        return data\n    except Exception as e:\n        print(f"Failed to load data: {e}")\n        return None'

In [43]:
def prepare_stock_data(file_path):
    """
    Prepare stock data from Excel/CSV file for risk analysis and portfolio management

    Args:
        file_path (str): Path to the Excel/CSV file

    Returns:
        dict: Dictionary of DataFrames with processed data for each stock
    """
    # Determine file type and read accordingly
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        df = pd.read_excel(file_path)
    else:  # Assume CSV
        df = pd.read_csv(file_path)

    print(f"Original data shape: {df.shape}")
    print(f"Available columns: {df.columns.tolist()}")

    # Check for date column - try different possible names
    date_column = None
    for possible_name in ['date', 'Date', 'DATE', 'datetime', 'Datetime', 'time', 'Time', 'timestamp', 'Timestamp']:
        if possible_name in df.columns:
            date_column = possible_name
            break

    if date_column is None:
        # If no date column found, create a simple date index
        print("Warning: No date column found. Creating a sequential date index.")
        df['date'] = pd.date_range(start='2022-01-01', periods=len(df), freq='D')
        date_column = 'date'

    # Convert date to datetime if not already
    if not pd.api.types.is_datetime64_any_dtype(df[date_column]):
        df[date_column] = pd.to_datetime(df[date_column])

    # Check for ticker column
    ticker_column = None
    for possible_name in ['tic', 'ticker', 'Ticker', 'symbol', 'Symbol', 'stock', 'Stock']:
        if possible_name in df.columns:
            ticker_column = possible_name
            break

    if ticker_column is None:
        # If no ticker column found, assume it's a single stock and create a default
        print("Warning: No ticker column found. Assuming single stock data with ticker 'STOCK'.")
        df['ticker'] = 'STOCK'
        ticker_column = 'ticker'

    # Sort by ticker and date
    df = df.sort_values([ticker_column, date_column])

    # Calculate additional features for risk assessment
    stock_dfs = {}
    skipped_stocks = 0

    # Process each stock separately
    for ticker in df[ticker_column].unique():
        stock_data = df[df[ticker_column] == ticker].copy()

        # Rename columns for consistency in the rest of the code
        if date_column != 'date':
            stock_data['date'] = stock_data[date_column]
        if ticker_column != 'tic':
            stock_data['tic'] = stock_data[ticker_column]

        # Ensure we have enough data (at least 1 year)
        if len(stock_data) < 252:  # Approx trading days in a year
            print(f"Skipping {ticker} - insufficient data ({len(stock_data)} rows)")
            skipped_stocks += 1
            continue

        # Check for price columns
        price_columns = {}
        for col_type in ['close', 'high', 'low', 'open']:
            for possible_name in [col_type, col_type.capitalize(), col_type.upper(),
                                 f"{col_type}_price", f"{col_type.capitalize()}_Price"]:
                if possible_name in stock_data.columns:
                    price_columns[col_type] = possible_name
                    break

        # Check for volume column
        volume_column = None
        for possible_name in ['volume', 'Volume', 'VOLUME', 'vol', 'Vol']:
            if possible_name in stock_data.columns:
                volume_column = possible_name
                break

        # Make sure we have at least a close price
        if 'close' not in price_columns:
            print(f"Warning: No close price column found for {ticker}. Skipping this stock.")
            continue

        # Calculate daily returns
        stock_data['daily_return'] = stock_data[price_columns['close']].pct_change()

        # Calculate volatility (20-day and 60-day)
        stock_data['volatility_20d'] = stock_data['daily_return'].rolling(window=20).std() * np.sqrt(252)
        stock_data['volatility_60d'] = stock_data['daily_return'].rolling(window=60).std() * np.sqrt(252)

        # Calculate drawdowns
        stock_data['cumulative_return'] = (1 + stock_data['daily_return']).cumprod()
        stock_data['rolling_max'] = stock_data['cumulative_return'].cummax()
        stock_data['drawdown'] = (stock_data['cumulative_return'] / stock_data['rolling_max']) - 1
        stock_data['max_drawdown_60d'] = stock_data['drawdown'].rolling(window=60).min()

        # Calculate trading volume metrics if volume data is available
        if volume_column:
            stock_data['volume_change'] = stock_data[volume_column].pct_change()
            stock_data['volume_ma10'] = stock_data[volume_column].rolling(window=10).mean()
            stock_data['relative_volume'] = stock_data[volume_column] / stock_data['volume_ma10']
        else:
            # Create dummy volume metrics if volume data is missing
            stock_data['volume_change'] = 0
            stock_data['volume_ma10'] = 100
            stock_data['relative_volume'] = 1

        # Calculate momentum indicators
        close_col = price_columns['close']
        stock_data['momentum_5d'] = stock_data[close_col].pct_change(periods=5)
        stock_data['momentum_20d'] = stock_data[close_col].pct_change(periods=20)

        # Calculate RSI (Relative Strength Index)
        delta = stock_data[close_col].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        # Avoid division by zero
        rs = gain / loss.replace(0, 0.001)
        stock_data['rsi_14d'] = 100 - (100 / (1 + rs))

        # Calculate Bollinger Bands
        stock_data['sma_20'] = stock_data[close_col].rolling(window=20).mean()
        stock_data['bollinger_std'] = stock_data[close_col].rolling(window=20).std()
        stock_data['bollinger_upper'] = stock_data['sma_20'] + 2 * stock_data['bollinger_std']
        stock_data['bollinger_lower'] = stock_data['sma_20'] - 2 * stock_data['bollinger_std']
        # Avoid division by zero
        stock_data['bollinger_width'] = (stock_data['bollinger_upper'] - stock_data['bollinger_lower']) / stock_data['sma_20'].replace(0, 0.001)

        # Calculate MACD
        stock_data['ema_12'] = stock_data[close_col].ewm(span=12, adjust=False).mean()
        stock_data['ema_26'] = stock_data[close_col].ewm(span=26, adjust=False).mean()
        stock_data['macd'] = stock_data['ema_12'] - stock_data['ema_26']
        stock_data['macd_signal'] = stock_data['macd'].ewm(span=9, adjust=False).mean()
        stock_data['macd_hist'] = stock_data['macd'] - stock_data['macd_signal']

        # Add standardized column names for key price data to ensure compatibility with rest of the code
        stock_data['close'] = stock_data[close_col]
        for col_type in ['high', 'low', 'open']:
            if col_type in price_columns:
                stock_data[col_type] = stock_data[price_columns[col_type]]
            else:
                # If missing, use close price as fallback
                stock_data[col_type] = stock_data[close_col]

        if volume_column:
            stock_data['volume'] = stock_data[volume_column]
        else:
            stock_data['volume'] = 0

        # Drop NaN values that result from calculations
        stock_data = stock_data.dropna()

        # Only include stock if we still have meaningful data after calculations
        if len(stock_data) >= 126:  # At least ~6 months of data
            stock_dfs[ticker] = stock_data
        else:
            skipped_stocks += 1

    print(f"Processed {len(stock_dfs)} stocks, skipped {skipped_stocks} stocks")
    return stock_dfs

In [31]:
def calculate_risk_metrics(stock_dfs, lookback_period=252):
    """
    Calculate risk metrics for each stock

    Args:
        stock_dfs (dict): Dictionary of stock DataFrames
        lookback_period (int): Period to use for calculations (default 1 year)

    Returns:
        DataFrame: Summary of risk metrics for all stocks
    """
    risk_metrics = []

    for ticker, data in stock_dfs.items():
        # Use recent data only
        recent_data = data.iloc[-lookback_period:] if len(data) > lookback_period else data

        # Skip stocks with zero or constant prices (no price movement)
        if recent_data['daily_return'].std() == 0 or recent_data['close'].nunique() <= 1:
            print(f"Skipping {ticker} - no price variation")
            continue

        # Calculate key risk metrics
        metrics = {
            'ticker': ticker,
            'avg_daily_return': recent_data['daily_return'].mean(),
            'annualized_return': recent_data['daily_return'].mean() * 252,
            'volatility': recent_data['daily_return'].std() * np.sqrt(252),
            'max_drawdown': recent_data['drawdown'].min(),
            'sharpe_ratio': (recent_data['daily_return'].mean() * 252) /
                           (recent_data['daily_return'].std() * np.sqrt(252))
                           if recent_data['daily_return'].std() > 0 else 0,
            'avg_volume': recent_data['volume'].mean(),
            'volume_volatility': recent_data['volume_change'].std() if not pd.isna(recent_data['volume_change'].std()) else 0,
            'momentum_5d': recent_data['momentum_5d'].mean(),
            'momentum_20d': recent_data['momentum_20d'].mean(),
            'avg_rsi': recent_data['rsi_14d'].mean(),
            'bollinger_width_avg': recent_data['bollinger_width'].mean(),
            'macd_histogram_avg': recent_data['macd_hist'].mean(),
            'data_points': len(recent_data)
        }

        # Add recent indicators (latest values)
        latest = recent_data.iloc[-1]
        metrics.update({
            'current_price': latest['close'],
            'current_volatility_20d': latest['volatility_20d'],
            'current_rsi': latest['rsi_14d'],
            'current_macd': latest['macd'],
            'current_macd_signal': latest['macd_signal'],
            'current_bollinger_position': (latest['close'] - latest['bollinger_lower']) /
                                        (latest['bollinger_upper'] - latest['bollinger_lower'])
        })

        risk_metrics.append(metrics)

    # Convert to DataFrame
    if not risk_metrics:
        print("No valid stocks with risk metrics found!")
        return pd.DataFrame()

    risk_df = pd.DataFrame(risk_metrics)

    # Handle extreme values and missing data
    for col in ['volatility', 'max_drawdown', 'volume_volatility', 'sharpe_ratio',
                'current_bollinger_position', 'current_rsi']:
        # Replace infinity values
        risk_df[col] = risk_df[col].replace([np.inf, -np.inf], np.nan)

        # Handle missing values with median imputation
        if risk_df[col].isna().any():
            median_val = risk_df[col].median()
            risk_df[col] = risk_df[col].fillna(median_val)

    # Calculate composite risk score
    vol_component = (risk_df['volatility'] / risk_df['volatility'].max()) if risk_df['volatility'].max() > 0 else 0
    dd_component = (risk_df['max_drawdown'].abs() / risk_df['max_drawdown'].abs().max()) if risk_df['max_drawdown'].abs().max() > 0 else 0
    vol_vol_component = (risk_df['volume_volatility'] / risk_df['volume_volatility'].max()) if risk_df['volume_volatility'].max() > 0 else 0

    # RSI component (extreme values indicate higher risk)
    rsi_component = ((risk_df['current_rsi'] - 50).abs() / 50)

    # Sharpe ratio is inversely related to risk (higher is better)
    sharpe_range = risk_df['sharpe_ratio'].max() - risk_df['sharpe_ratio'].min()
    if sharpe_range > 0:
        sharpe_component = 1 - ((risk_df['sharpe_ratio'] - risk_df['sharpe_ratio'].min()) / sharpe_range)
    else:
        sharpe_component = 0.5  # Neutral value if all stocks have the same Sharpe ratio

    # Add weighted risk score calculation with additional indicators
    risk_df['risk_score'] = (
        vol_component * 0.35 +
        dd_component * 0.25 +
        vol_vol_component * 0.10 +
        sharpe_component * 0.15 +
        rsi_component * 0.15
    )

    # Ensure all stocks have a valid risk score
    if risk_df['risk_score'].isna().any():
        print(f"Warning: {risk_df['risk_score'].isna().sum()} stocks have NaN risk scores")
        # Assign median risk score to stocks with NaN
        risk_df['risk_score'] = risk_df['risk_score'].fillna(risk_df['risk_score'].median())

    # Ensure we don't have exactly 0 or 1 risk scores (adjust slightly if needed)
    risk_df.loc[risk_df['risk_score'] == 0, 'risk_score'] = 0.001
    risk_df.loc[risk_df['risk_score'] == 1, 'risk_score'] = 0.999

    # Classify risk levels using qcut for equal-sized groups
    try:
        risk_df['risk_category'] = pd.qcut(
            risk_df['risk_score'],
            q=3,
            labels=['Low', 'Medium', 'High']
        )
    except ValueError:
        # Fall back to manual classification if qcut fails
        risk_thresholds = [0, 0.33, 0.67, 1.0]
        risk_df['risk_category'] = pd.cut(
            risk_df['risk_score'],
            bins=risk_thresholds,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

    return risk_df

In [32]:
class PortfolioRiskEnv(gym.Env):
    """Custom Environment for portfolio risk management using SAC"""
    metadata = {'render.modes': ['human']}

    def __init__(self, stock_data_dict, risk_df, feature_columns, initial_capital=1000000):
        super(PortfolioRiskEnv, self).__init__()

        # Store data
        self.stock_data_dict = stock_data_dict
        self.risk_df = risk_df
        self.feature_columns = feature_columns
        self.tickers = list(stock_data_dict.keys())
        self.initial_capital = initial_capital

        # Portfolio parameters
        self.max_position_size = 0.2  # Maximum allocation to a single asset (20%)
        self.transaction_fee = 0.001  # 0.1% transaction fee
        self.max_stocks_in_portfolio = 20
        self.risk_free_rate = 0.02  # 2% annual risk-free rate

        # Environment parameters
        self.window_size = 30  # Historical window to consider
        self.max_steps = 252  # Approximately 1 trading year

        # Define action and observation spaces
        # Action space: For each stock, we have a continuous action between -1 and 1
        # -1 means sell all, 0 means hold current position, 1 means buy up to max allocation
        # Plus one more action for overall risk budget allocation
        self.action_space = spaces.Box(
            low=-1,
            high=1,
            shape=(min(len(self.tickers), self.max_stocks_in_portfolio) + 1,),
            dtype=np.float32
        )

        # Observation space (per stock, per feature, plus portfolio stats)
        stock_features = len(feature_columns) * 5  # 5 statistics per feature (mean, std, min, max, last)
        portfolio_features = 10  # Portfolio level features like total value, cash, etc.
        self.observation_space = spaces.Box(
            low=-10.0,
            high=10.0,
            shape=(min(len(self.tickers), self.max_stocks_in_portfolio) * stock_features + portfolio_features,),
            dtype=np.float32
        )

        # Initialize portfolio state
        self.reset()

    def reset(self, seed=None, options=None):
        """Reset the environment to start a new episode"""
        super().reset(seed=seed)

        # Select a subset of stocks for this episode, preferring stocks with full data
        valid_tickers = [ticker for ticker in self.tickers
                        if len(self.stock_data_dict[ticker]) >= self.window_size + self.max_steps]

        if len(valid_tickers) >= self.max_stocks_in_portfolio:
            self.episode_tickers = np.random.choice(valid_tickers, self.max_stocks_in_portfolio, replace=False)
        else:
            self.episode_tickers = valid_tickers

        # Initialize portfolio
        self.cash = self.initial_capital
        self.positions = {ticker: 0 for ticker in self.episode_tickers}
        self.position_values = {ticker: 0 for ticker in self.episode_tickers}
        self.portfolio_value_history = [self.initial_capital]
        self.risk_budget = 1.0  # Full risk budget at start

        # Initialize current step
        self.current_step = 0

        # For each stock, select a random start point that ensures we have enough future data
        self.stock_time_pointers = {}
        for ticker in self.episode_tickers:
            data = self.stock_data_dict[ticker]
            max_start = len(data) - self.window_size - self.max_steps
            if max_start <= 0:
                max_start = 0  # Use beginning if not enough data
            self.stock_time_pointers[ticker] = np.random.randint(self.window_size, max_start + self.window_size)

        # Get initial observation
        observation = self._get_observation()

        return observation, {}

    def step(self, action):
        """Take an action in the environment"""
        self.current_step += 1

        # Parse actions
        risk_budget_action = action[0]  # First action controls overall risk budget
        stock_actions = action[1:]  # Remaining actions control individual stock allocations

        # Adjust risk budget (between 0.5 and 1.5 of base risk)
        self.risk_budget = 1.0 + (risk_budget_action * 0.5)  # Scale to range [0.5, 1.5]

        # Calculate current portfolio value
        current_portfolio_value = self.cash + sum(self.position_values.values())

        # Execute trades based on actions
        reward = self._execute_trades(stock_actions, current_portfolio_value)

        # Advance time for all stocks
        self._advance_time()

        # Update position values based on new prices
        self._update_position_values()

        # Calculate new portfolio value
        new_portfolio_value = self.cash + sum(self.position_values.values())

        # Record portfolio value history
        self.portfolio_value_history.append(new_portfolio_value)

        # Calculate portfolio performance metrics
        portfolio_return = (new_portfolio_value / current_portfolio_value) - 1

        # Additional reward components
        reward += self._calculate_risk_adjusted_reward(portfolio_return)

        # Check if episode is done
        done = self.current_step >= self.max_steps

        # Get next observation
        next_obs = self._get_observation()

        # Information dictionary
        info = {
            'portfolio_value': new_portfolio_value,
            'portfolio_return': portfolio_return,
            'cash': self.cash,
            'risk_budget': self.risk_budget,
            'positions': self.positions.copy(),
            'position_values': self.position_values.copy()
        }

        return next_obs, reward, done, False, info

    def _execute_trades(self, stock_actions, current_portfolio_value):
        """Execute trades based on actions"""
        reward = 0

        # Scale actions by risk budget
        risk_weighted_actions = stock_actions * self.risk_budget

        # Calculate target values for each position
        target_values = {}
        for i, ticker in enumerate(self.episode_tickers):
            if i < len(risk_weighted_actions):
                # Convert action to target position size (% of portfolio)
                # Actions in range [-1, 1] get converted to [0, max_position_size]
                action = risk_weighted_actions[i]
                target_allocation = (action + 1) / 2 * self.max_position_size
                target_values[ticker] = target_allocation * current_portfolio_value
            else:
                target_values[ticker] = 0

        # Execute trades to achieve target values
        for ticker in self.episode_tickers:
            if ticker in target_values:
                current_price = self._get_current_price(ticker)

                # Skip if price data is missing
                if current_price <= 0:
                    continue

                current_value = self.position_values.get(ticker, 0)
                target_value = target_values[ticker]

                # Calculate shares to trade
                value_difference = target_value - current_value
                shares_to_trade = value_difference / current_price

                # Calculate transaction fee
                transaction_fee = abs(value_difference) * self.transaction_fee

                # Execute trade if it's significant enough
                if abs(value_difference) > 100:  # Minimum trade size
                    if shares_to_trade > 0:  # Buy
                        cost = value_difference + transaction_fee
                        if self.cash >= cost:
                            self.positions[ticker] = self.positions.get(ticker, 0) + shares_to_trade
                            self.position_values[ticker] = current_value + value_difference
                            self.cash -= cost

                            # Small reward for diversification
                            if self.positions[ticker] > 0 and current_value == 0:
                                reward += 0.02
                    else:  # Sell
                        proceeds = -value_difference - transaction_fee
                        current_shares = self.positions.get(ticker, 0)
                        if current_shares >= abs(shares_to_trade):
                            self.positions[ticker] = current_shares - abs(shares_to_trade)
                            self.position_values[ticker] = current_value - abs(value_difference)
                            self.cash += proceeds

                # Small penalty for large transactions (to encourage gradual position building)
                penalty = abs(value_difference) / current_portfolio_value * 0.01
                reward -= min(penalty, 0.1)  # Cap the penalty

        return reward

    def _advance_time(self):
        """Advance time by one step for all stocks"""
        for ticker in self.episode_tickers:
            self.stock_time_pointers[ticker] += 1

    def _update_position_values(self):
        """Update the value of all positions based on current prices"""
        for ticker in self.episode_tickers:
            current_price = self._get_current_price(ticker)
            if current_price > 0:
                self.position_values[ticker] = self.positions[ticker] * current_price

    def _get_current_price(self, ticker):
        """Get the current price of a stock"""
        try:
            data = self.stock_data_dict[ticker]
            idx = min(self.stock_time_pointers[ticker], len(data) - 1)
            return data.iloc[idx]['close']
        except (IndexError, KeyError):
            print(f"Warning: Cannot get price for {ticker} at index {self.stock_time_pointers.get(ticker)}")
            return 0

    def _calculate_risk_adjusted_reward(self, portfolio_return):
        """Calculate risk-adjusted reward components"""
        # Base reward is the portfolio return
        reward = portfolio_return * 10  # Scale up returns to make rewards more meaningful

        # Calculate volatility penalty if we have enough history
        if len(self.portfolio_value_history) > 20:
            returns = np.diff(self.portfolio_value_history[-20:]) / self.portfolio_value_history[-21:-1]
            volatility = np.std(returns) * np.sqrt(252)  # Annualized volatility

            # Sharpe ratio component (risk-adjusted return)
            # Use a smoothed version to avoid extreme values
            excess_return = portfolio_return * 252 - self.risk_free_rate  # Annualized excess return
            sharpe = excess_return / (volatility + 0.001)  # Add small constant to avoid division by zero

            # Add sharpe ratio component to reward
            reward += np.clip(sharpe * 0.1, -0.5, 0.5)

            # Drawdown penalty
            rolling_max = np.maximum.accumulate(self.portfolio_value_history[-20:])
            drawdown = (self.portfolio_value_history[-1] / rolling_max[-1]) - 1

            # Penalize drawdowns more heavily as they get larger
            drawdown_penalty = abs(drawdown) ** 2 * 5 if drawdown < 0 else 0
            reward -= drawdown_penalty

        # Risk budget utilization reward
        # Reward for using risk budget effectively (higher returns per unit of risk)
        risk_efficiency = portfolio_return / (self.risk_budget + 0.01)  # Add small constant to avoid division by zero
        reward += np.clip(risk_efficiency * 5, -0.5, 0.5)

        # Clip total reward to avoid extreme values
        return np.clip(reward, -10, 10)

    def _get_observation(self):
        """Get the current observation (state)"""
        observations = []

        # For each stock in the portfolio
        for ticker in self.episode_tickers:
            stock_features = []
            data = self.stock_data_dict[ticker]

            # Get historical window of data
            start_idx = max(0, self.stock_time_pointers[ticker] - self.window_size)
            end_idx = self.stock_time_pointers[ticker]

            if start_idx >= len(data) or end_idx >= len(data) or start_idx >= end_idx:
                # If we don't have valid data, add zeros
                flat_features = np.zeros(len(self.feature_columns) * 5)
            else:
                window_data = data.iloc[start_idx:end_idx]

                # Extract features
                for col in self.feature_columns:
                    if col in window_data.columns:
                        values = window_data[col].values

                        # Replace NaN and Inf values
                        values = np.nan_to_num(values, nan=0.0, posinf=10.0, neginf=-10.0)

                        if len(values) > 0:
                            feature_mean = np.mean(values)
                            feature_std = np.std(values)
                            feature_min = np.min(values)
                            feature_max = np.max(values)
                            feature_last = values[-1]
                        else:
                            feature_mean = feature_std = feature_min = feature_max = feature_last = 0.0

                        # Normalize features
                        feature_mean = np.clip(feature_mean, -10.0, 10.0)
                        feature_std = np.clip(feature_std, 0.0, 10.0)
                        feature_min = np.clip(feature_min, -10.0, 10.0)
                        feature_max = np.clip(feature_max, -10.0, 10.0)
                        feature_last = np.clip(feature_last, -10.0, 10.0)

                        stock_features.extend([feature_mean, feature_std, feature_min, feature_max, feature_last])
                    else:
                        # Pad with zeros if feature not available
                        stock_features.extend([0.0, 0.0, 0.0, 0.0, 0.0])

                # Add risk information
                if ticker in self.risk_df['ticker'].values:
                    risk_info = self.risk_df[self.risk_df['ticker'] == ticker].iloc[0]
                    risk_score = risk_info['risk_score']
                    volatility = risk_info['volatility']
                    sharpe = risk_info['sharpe_ratio']
                else:
                    risk_score = 0.5  # Medium risk if not found
                    volatility = 0.2  # Moderate volatility
                    sharpe = 0.0  # Neutral Sharpe ratio

                stock_features.extend([risk_score, volatility, sharpe])

                # Add position information
                current_price = self._get_current_price(ticker)
                position_size = self.positions.get(ticker, 0)
                position_value = position_size * current_price
                allocation = position_value / (self.cash + sum(self.position_values.values())) if (self.cash + sum(self.position_values.values())) > 0 else 0

                stock_features.extend([position_size, position_value, allocation])

            # Ensure fixed length for all stocks
            if len(stock_features) < len(self.feature_columns) * 5:
                padding = [0.0] * (len(self.feature_columns) * 5 - len(stock_features))
                stock_features.extend(padding)
            elif len(stock_features) > len(self.feature_columns) * 5:
                stock_features = stock_features[:len(self.feature_columns) * 5]

            observations.extend(stock_features)

        # Add portfolio-level features
        portfolio_value = self.cash + sum(self.position_values.values())
        portfolio_features = [
            self.cash / self.initial_capital,  # Normalized cash
            portfolio_value / self.initial_capital,  # Normalized portfolio value
            self.risk_budget,  # Current risk budget
            len([t for t in self.positions if self.positions[t] > 0]) / len(self.episode_tickers),  # Portfolio diversification
            self.current_step / self.max_steps  # Episode progress
        ]

        # Add recent portfolio performance if available
        if len(self.portfolio_value_history) > 5:
            recent_returns = np.diff(self.portfolio_value_history[-5:]) / self.portfolio_value_history[-6:-1]
            portfolio_features.extend([
                np.mean(recent_returns),  # Average recent return
                np.std(recent_returns) * np.sqrt(252),  # Recent volatility (annualized)
                np.min(recent_returns),  # Worst recent return
                np.max(recent_returns),  # Best recent return
                (portfolio_value / np.max(self.portfolio_value_history)) - 1  # Current drawdown
            ])
        else:
            portfolio_features.extend([0.0, 0.0, 0.0, 0.0, 0.0])

        observations.extend(portfolio_features)

        # Ensure fixed length of the observation vector
        expected_length = min(len(self.tickers), self.max_stocks_in_portfolio) * len(self.feature_columns) * 5 + 10

        if len(observations) < expected_length:
            padding = [0.0] * (expected_length - len(observations))
            observations.extend(padding)
        elif len(observations) > expected_length:
            observations = observations[:expected_length]

        # Final check for NaN/Inf values
        observations = np.array(observations, dtype=np.float32)
        observations = np.nan_to_num(observations, nan=0.0, posinf=10.0, neginf=-10.0)

        return observations

    def render(self):
        """Render the environment"""
        portfolio_value = self.cash + sum(self.position_values.values())
        print(f"Step: {self.current_step}/{self.max_steps}, Portfolio Value: ${portfolio_value:.2f}, Cash: ${self.cash:.2f}")
        print(f"Risk Budget: {self.risk_budget:.2f}")
        print("Current Positions:")
        for ticker in self.episode_tickers:
            if self.positions.get(ticker, 0) > 0:
                print(f"  {ticker}: {self.positions[ticker]:.2f} shares, ${self.position_values.get(ticker, 0):.2f}")



In [34]:
def train_sac_model(stock_dfs, risk_df, total_timesteps=200000):
    """
    Train a SAC model for portfolio risk management

    Args:
        stock_dfs: Dictionary of processed stock DataFrames
        risk_df: DataFrame with risk metrics for all stocks
        total_timesteps: Total training timesteps

    Returns:
        Trained SAC model
    """
    # Define feature columns from available columns
    sample_df = next(iter(stock_dfs.values()))
    available_cols = sample_df.columns.tolist()

    # Select features that exist in the data
    required_features = [
        'daily_return', 'volatility_20d', 'drawdown', 'volume_change',
        'momentum_5d', 'momentum_20d', 'rsi_14d', 'macd', 'bollinger_width'
    ]
    feature_columns = [col for col in required_features if col in available_cols]

    if not feature_columns:
        raise ValueError("No required features found in the data. Make sure you have calculated these features.")

    print(f"Using features: {feature_columns}")

    # Create and wrap the environment
    def make_env():
        try:
            env = PortfolioRiskEnv(stock_dfs, risk_df, feature_columns)
            env = Monitor(env)
            return env
        except Exception as e:
            print(f"Error creating environment: {str(e)}")
            raise

    # Create environment with error handling
    try:
        env = DummyVecEnv([make_env])

        # Test the environment
        obs = env.reset()
        test_action = env.action_space.sample()
        next_obs, reward, done, info = env.step([test_action])

        print("Environment test successful")

        # Create SAC model with hyperparameters tuned for portfolio management
        model = SAC(
            "MlpPolicy",
            env,
            learning_rate=0.0003,
            buffer_size=100000,      # Replay buffer size
            batch_size=256,          # Minibatch size
            ent_coef='auto',         # Entropy coefficient (auto-adjusts)
            gamma=0.99,              # Discount factor
            tau=0.005,               # Target network update rate
            train_freq=1,            # Update the model every train_freq steps
            gradient_steps=1,        # How many gradient steps to do after each rollout
            learning_starts=10000,   # Start learning after this many steps
            policy_kwargs=dict(
                net_arch=dict(
                    pi=[256, 256, 128],   # Policy network
                    qf=[256, 256, 128]    # Q-function network
                ),
                activation_fn=torch.nn.ReLU
            ),
            verbose=1,
            tensorboard_log="./sac_portfolio_risk_tensorboard/"
        )

        # Set up callbacks
        # Checkpoint callback to save model periodically
        checkpoint_callback = CheckpointCallback(
            save_freq=10000,
            save_path="./models/",
            name_prefix="sac_portfolio_risk"
        )

        # Create an evaluation environment
        eval_env = DummyVecEnv([make_env])

        # Evaluation callback
        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path="./models/",
            log_path="./logs/",
            eval_freq=10000,
            deterministic=True,
            render=False
        )

        # Train the model with progress tracking
        print(f"Starting SAC training for {total_timesteps} timesteps...")

        # Track progress manually in smaller chunks for stability
        log_interval = total_timesteps // 10
        for i in range(10):
            model.learn(
                total_timesteps=log_interval,
                callback=[checkpoint_callback, eval_callback],
                reset_num_timesteps=False if i > 0 else True,
                tb_log_name="SAC_portfolio_risk"
            )
            print(f"Completed {(i+1)*log_interval}/{total_timesteps} timesteps ({(i+1)*10}%)")

        print("SAC training completed")
        return model, env, feature_columns

    except Exception as e:
        print(f"Error during SAC setup/training: {str(e)}")
        raise

In [35]:
def evaluate_sac_model(model, stock_dfs, risk_df, feature_columns, n_episodes=10):
    """
    Evaluate a trained SAC model on the portfolio risk management task

    Args:
        model: Trained SAC model
        stock_dfs: Dictionary of processed stock DataFrames
        risk_df: DataFrame with risk metrics for all stocks
        feature_columns: Features used by the model
        n_episodes: Number of evaluation episodes

    Returns:
        dict: Evaluation metrics
    """
    # Create evaluation environment
    eval_env = PortfolioRiskEnv(stock_dfs, risk_df, feature_columns)

    # Track metrics
    all_returns = []
    all_sharpe_ratios = []
    all_max_drawdowns = []
    all_portfolio_values = []

    for episode in range(n_episodes):
        print(f"Evaluating episode {episode+1}/{n_episodes}")
        obs, _ = eval_env.reset()
        done = False
        step = 0

        # Initial portfolio value
        portfolio_values = [eval_env.cash + sum(eval_env.position_values.values())]

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)

            # Record portfolio value
            portfolio_values.append(info['portfolio_value'])

            step += 1

        # Calculate evaluation metrics
        final_return = (portfolio_values[-1] / portfolio_values[0]) - 1

        # Calculate returns from portfolio values
        returns = np.diff(portfolio_values) / portfolio_values[:-1]

        # Sharpe ratio (annualized)
        volatility = np.std(returns) * np.sqrt(252)
        sharpe_ratio = (np.mean(returns) * 252 - 0.02) / (volatility if volatility > 0 else 1e-6)

        # Maximum drawdown
        peak = np.maximum.accumulate(portfolio_values)
        drawdowns = (portfolio_values / peak) - 1
        max_drawdown = np.min(drawdowns)

        # Record metrics
        all_returns.append(final_return)
        all_sharpe_ratios.append(sharpe_ratio)
        all_max_drawdowns.append(max_drawdown)
        all_portfolio_values.append(portfolio_values)

        print(f"  Return: {final_return:.2%}, Sharpe: {sharpe_ratio:.2f}, Max Drawdown: {max_drawdown:.2%}")

    # Calculate average metrics
    avg_return = np.mean(all_returns)
    avg_sharpe = np.mean(all_sharpe_ratios)
    avg_max_drawdown = np.mean(all_max_drawdowns)

    print("\nEvaluation Summary:")
    print(f"Average Return: {avg_return:.2%}")
    print(f"Average Sharpe Ratio: {avg_sharpe:.2f}")
    print(f"Average Maximum Drawdown: {avg_max_drawdown:.2%}")

    return {
        'avg_return': avg_return,
        'avg_sharpe': avg_sharpe,
        'avg_max_drawdown': avg_max_drawdown,
        'all_returns': all_returns,
        'all_sharpe_ratios': all_sharpe_ratios,
        'all_max_drawdowns': all_max_drawdowns,
        'all_portfolio_values': all_portfolio_values
    }

In [36]:
def generate_portfolio_recommendations(model, stock_dfs, risk_df, feature_columns, risk_profile='Medium'):
    """
    Generate portfolio recommendations based on the trained SAC model

    Args:
        model: Trained SAC model
        stock_dfs: Dictionary of processed stock DataFrames
        risk_df: DataFrame with risk metrics for all stocks
        feature_columns: Features used by the model
        risk_profile: Risk tolerance ('Low', 'Medium', or 'High')

    Returns:
        DataFrame: Portfolio recommendations
    """
    # Create environment for generating recommendations
    env = PortfolioRiskEnv(stock_dfs, risk_df, feature_columns)

    # Set risk budget based on risk profile
    if risk_profile == 'Low':
        risk_budget_multiplier = 0.7  # 70% of normal risk
    elif risk_profile == 'Medium':
        risk_budget_multiplier = 1.0  # Normal risk
    else:  # High
        risk_budget_multiplier = 1.3  # 130% of normal risk

    # Reset environment
    obs, _ = env.reset()

    # Override risk budget if needed
    env.risk_budget = risk_budget_multiplier

    # Get model recommendation (action)
    action, _ = model.predict(obs, deterministic=True)

    # First action controls risk budget, remaining actions control stock allocations
    risk_budget_action = action[0]
    stock_actions = action[1:]

    # Current portfolio value
    portfolio_value = env.cash + sum(env.position_values.values())

    # Translate actions to target allocations
    target_allocations = {}
    for i, ticker in enumerate(env.episode_tickers):
        if i < len(stock_actions):
            # Convert action to target position size (% of portfolio)
            action_value = stock_actions[i]
            target_allocation = (action_value + 1) / 2 * env.max_position_size
            target_allocations[ticker] = target_allocation

    # Get current prices
    current_prices = {ticker: env._get_current_price(ticker) for ticker in env.episode_tickers}

    # Create recommendations dataframe
    recommendations = []
    for ticker in env.episode_tickers:
        if ticker in target_allocations and current_prices[ticker] > 0:
            # Get risk info
            if ticker in risk_df['ticker'].values:
                risk_info = risk_df[risk_df['ticker'] == ticker].iloc[0]
                risk_score = risk_info['risk_score']
                volatility = risk_info['volatility']
                sharpe = risk_info['sharpe_ratio']
                risk_category = risk_info['risk_category']
            else:
                risk_score = 0.5
                volatility = 0.2
                sharpe = 0.0
                risk_category = 'Medium'

            # Calculate target dollar amount and number of shares
            target_allocation = target_allocations[ticker]
            target_dollar = target_allocation * portfolio_value
            target_shares = target_dollar / current_prices[ticker]

            recommendations.append({
                'ticker': ticker,
                'allocation_pct': target_allocation * 100,
                'target_value': target_dollar,
                'target_shares': target_shares,
                'current_price': current_prices[ticker],
                'risk_score': risk_score,
                'risk_category': risk_category,
                'volatility': volatility,
                'sharpe_ratio': sharpe
            })

    # Create DataFrame and sort by allocation
    recommendations_df = pd.DataFrame(recommendations)
    if not recommendations_df.empty:
        recommendations_df = recommendations_df.sort_values('allocation_pct', ascending=False)

    return recommendations_df

In [37]:


def visualize_portfolio_performance(eval_results):
    """
    Visualize portfolio performance metrics from evaluation

    Args:
        eval_results: Results from evaluate_sac_model
    """
    # Create figure with multiple subplots
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Portfolio values over time (average and individual episodes)
    ax = axs[0, 0]
    for i, portfolio_values in enumerate(eval_results['all_portfolio_values']):
        steps = range(len(portfolio_values))
        ax.plot(steps, portfolio_values, alpha=0.3, linewidth=1)

    # Calculate and plot average portfolio value
    max_length = max(len(pv) for pv in eval_results['all_portfolio_values'])
    avg_portfolio = np.zeros(max_length)
    count = np.zeros(max_length)

    for pv in eval_results['all_portfolio_values']:
        avg_portfolio[:len(pv)] += pv
        count[:len(pv)] += 1

    avg_portfolio = avg_portfolio / np.maximum(count, 1)  # Avoid division by zero
    ax.plot(range(max_length), avg_portfolio, 'r-', linewidth=2, label='Average')

    ax.set_title('Portfolio Value Over Time')
    ax.set_xlabel('Steps')
    ax.set_ylabel('Portfolio Value ($)')
    ax.legend()
    ax.grid(True)

    # 2. Returns distribution
    ax = axs[0, 1]
    ax.hist(eval_results['all_returns'], bins=10, alpha=0.7)
    ax.axvline(eval_results['avg_return'], color='r', linestyle='dashed', linewidth=2,
               label=f"Avg: {eval_results['avg_return']:.2%}")
    ax.set_title('Distribution of Returns')
    ax.set_xlabel('Return')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True)

    # 3. Sharpe ratios
    ax = axs[1, 0]
    ax.hist(eval_results['all_sharpe_ratios'], bins=10, alpha=0.7)
    ax.axvline(eval_results['avg_sharpe'], color='r', linestyle='dashed', linewidth=2,
               label=f"Avg: {eval_results['avg_sharpe']:.2f}")
    ax.set_title('Distribution of Sharpe Ratios')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True)

    # 4. Drawdowns
    ax = axs[1, 1]
    ax.hist(eval_results['all_max_drawdowns'], bins=10, alpha=0.7)
    ax.axvline(eval_results['avg_max_drawdown'], color='r', linestyle='dashed', linewidth=2,
               label=f"Avg: {eval_results['avg_max_drawdown']:.2%}")
    ax.set_title('Distribution of Maximum Drawdowns')
    ax.set_xlabel('Maximum Drawdown')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True)

    plt.tight_layout()
    plt.savefig('portfolio_performance.png')
    plt.show()

In [38]:
def calculate_risk_metrics(stock_dfs, lookback_period=None):
    """
    Calculate risk metrics for each stock with flexible lookback period

    Args:
        stock_dfs (dict): Dictionary of stock DataFrames
        lookback_period (int): Period to use for calculations (if None, uses all available data)

    Returns:
        DataFrame: Summary of risk metrics for all stocks
    """
    risk_metrics = []

    for ticker, data in stock_dfs.items():
        # Adjust lookback period based on available data
        if lookback_period is None or lookback_period >= len(data):
            recent_data = data  # Use all available data
        else:
            recent_data = data.iloc[-lookback_period:]

        # Skip stocks with zero or constant prices (no price movement)
        if recent_data['close'].nunique() <= 1:
            print(f"Skipping {ticker} - no price variation")
            continue

        # Calculate key risk metrics with error handling
        try:
            metrics = {
                'ticker': ticker,
                'avg_daily_return': recent_data['daily_return'].mean(),
                'annualized_return': recent_data['daily_return'].mean() * 252,
                'volatility': recent_data['daily_return'].std() * np.sqrt(252) if len(recent_data) > 1 else 0.2,
                'max_drawdown': recent_data['drawdown'].min() if 'drawdown' in recent_data else -0.1,
                'sharpe_ratio': (recent_data['daily_return'].mean() * 252) /
                               (recent_data['daily_return'].std() * np.sqrt(252))
                               if recent_data['daily_return'].std() > 0 and len(recent_data) > 1 else 0,
                'avg_volume': recent_data['volume'].mean() if 'volume' in recent_data else 100,
                'data_points': len(recent_data)
            }

            # Add additional metrics if available
            for field in ['volume_change', 'momentum_5d', 'momentum_20d', 'rsi_14d',
                         'bollinger_width', 'macd_hist']:
                if field in recent_data:
                    metrics[field.lower() + '_avg'] = recent_data[field].mean()
                else:
                    metrics[field.lower() + '_avg'] = 0

            # Add recent indicators (latest values) if available
            if len(recent_data) > 0:
                latest = recent_data.iloc[-1]
                metrics.update({
                    'current_price': latest['close'],
                    'current_volatility': latest['volatility_20d'] if 'volatility_20d' in latest else 0.15,
                    'current_rsi': latest['rsi_14d'] if 'rsi_14d' in latest else 50,
                    'current_macd': latest['macd'] if 'macd' in latest else 0,
                })

            risk_metrics.append(metrics)
        except Exception as e:
            print(f"Error calculating metrics for {ticker}: {str(e)}")
            continue

    # Convert to DataFrame
    if not risk_metrics:
        print("No valid stocks with risk metrics found!")

        # Create a simple dummy risk_df with one stock if no valid stocks
        if len(stock_dfs) > 0:
            ticker = list(stock_dfs.keys())[0]
            risk_metrics = [{
                'ticker': ticker,
                'risk_score': 0.5,
                'risk_category': 'Medium',
                'volatility': 0.15,
                'max_drawdown': -0.1,
                'sharpe_ratio': 0.5,
                'avg_daily_return': 0.001,
                'annualized_return': 0.252,
                'current_price': 100,
                'current_volatility': 0.15,
                'current_rsi': 50,
                'current_macd': 0,
                'data_points': len(stock_dfs[ticker])
            }]

        return pd.DataFrame(risk_metrics)

    risk_df = pd.DataFrame(risk_metrics)

    # Handle extreme values and missing data
    numeric_columns = risk_df.select_dtypes(include=['number']).columns
    for col in numeric_columns:
        # Replace infinity values
        risk_df[col] = risk_df[col].replace([np.inf, -np.inf], np.nan)

        # Handle missing values with median imputation
        if risk_df[col].isna().any():
            median_val = risk_df[col].median()
            if pd.isna(median_val):  # If median is also NaN
                median_val = 0  # Use 0 as fallback
            risk_df[col] = risk_df[col].fillna(median_val)

    # Calculate simplified risk score if we have limited data
    if 'volatility' in risk_df.columns and 'max_drawdown' in risk_df.columns:
        # Normalize components
        vol_component = (risk_df['volatility'] / risk_df['volatility'].max()) if risk_df['volatility'].max() > 0 else 0.5
        dd_component = (risk_df['max_drawdown'].abs() / risk_df['max_drawdown'].abs().max()) if risk_df['max_drawdown'].abs().max() > 0 else 0.5

        # Simple risk score (higher volatility and deeper drawdowns = higher risk)
        risk_df['risk_score'] = (vol_component * 0.6 + dd_component * 0.4).clip(0.001, 0.999)
    else:
        # Fallback if we don't have proper metrics
        risk_df['risk_score'] = 0.5  # Neutral risk score

    # Classify risk levels
    try:
        # Try equal-sized groups first
        risk_df['risk_category'] = pd.qcut(
            risk_df['risk_score'],
            q=3,
            labels=['Low', 'Medium', 'High']
        )
    except ValueError:
        # Fall back to manual classification if qcut fails
        risk_df['risk_category'] = pd.cut(
            risk_df['risk_score'],
            bins=[0, 0.33, 0.67, 1.0],
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

    return risk_df

In [39]:
def main():
    """Main execution function"""
    # Hard-coded file path - replace with your actual file path
    file_path = "/content/stock_risk_analysis.csv"  # <-- CHANGE THIS to your file path

    print(f"Using data file: {file_path}")

    # Process data with reduced minimum data requirement
    stock_dfs = prepare_stock_data(file_path, min_data_points=1)

    if len(stock_dfs) == 0:
        print("No valid stock data found. Cannot continue.")
        return

    # Calculate risk metrics with flexible lookback
    risk_df = calculate_risk_metrics(stock_dfs, lookback_period=None)

    # Train SAC model with reduced expectations
    try:
        # Use a smaller timestep for faster training
        total_timesteps = 50000  # Reduced from 100000

        # Define feature columns from available columns
        sample_df = next(iter(stock_dfs.values()))
        available_cols = sample_df.columns.tolist()

        # Select features that exist in the data
        required_features = [
            'daily_return', 'close', 'volume'  # Minimum required features
        ]

        # Add optional features if available
        optional_features = [
            'volatility_20d', 'drawdown', 'volume_change',
            'momentum_5d', 'momentum_20d', 'rsi_14d', 'macd', 'bollinger_width'
        ]

        feature_columns = [col for col in required_features if col in available_cols]
        feature_columns += [col for col in optional_features if col in available_cols]

        if len(feature_columns) < 3:
            raise ValueError(f"Insufficient features available: {feature_columns}. Need at least 3 features.")

        print(f"Using features: {feature_columns}")

        model, env, feature_columns = train_sac_model(stock_dfs, risk_df, total_timesteps=total_timesteps)

        # Save trained model
        model.save("sac_portfolio_risk_model")
        print("Model saved successfully")

        # Evaluate model
        eval_results = evaluate_sac_model(model, stock_dfs, risk_df, feature_columns, n_episodes=5)

        # Visualize results
        visualize_portfolio_performance(eval_results)

        # Generate recommendations for different risk profiles
        for risk_profile in ['Low', 'Medium', 'High']:
            recommendations = generate_portfolio_recommendations(
                model, stock_dfs, risk_df, feature_columns, risk_profile=risk_profile
            )

            print(f"\n{risk_profile} Risk Portfolio Recommendations:")
            if not recommendations.empty:
                display_cols = [col for col in ['ticker', 'allocation_pct', 'risk_category', 'sharpe_ratio']
                               if col in recommendations.columns]
                print(recommendations[display_cols].head(10))

                # Save recommendations to CSV
                recommendations.to_csv(f"{risk_profile.lower()}_risk_portfolio_recommendations.csv", index=False)
            else:
                print("No recommendations generated")

    except Exception as e:
        print(f"Error in SAC model processing: {str(e)}")
        print("Falling back to traditional risk classification only")

        # Print the available risk classifications as fallback
        if not risk_df.empty:
            print("\nRisk Classification Results:")
            display_cols = [col for col in ['ticker', 'risk_category', 'risk_score', 'volatility', 'max_drawdown']
                           if col in risk_df.columns]
            print(risk_df[display_cols].head(20))
            risk_df.to_csv("stock_risk_classification.csv", index=False)

In [41]:
def main():
    """Main execution function"""
    # Hard-coded file path - replace with your actual file path
    file_path = "/content/stock_risk_analysis.csv"  # <-- CHANGE THIS to your file path

    print(f"Using data file: {file_path}")

    # Process data
    stock_dfs = prepare_stock_data(file_path)

    # Calculate risk metrics
    risk_df = calculate_risk_metrics(stock_dfs)

    if not risk_df.empty:
        # Train SAC model
        try:
            model, env, feature_columns = train_sac_model(stock_dfs, risk_df, total_timesteps=100000)

            # Save trained model
            model.save("sac_portfolio_risk_model")
            print("Model saved successfully")

            # Evaluate model
            eval_results = evaluate_sac_model(model, stock_dfs, risk_df, feature_columns)

            # Visualize results
            visualize_portfolio_performance(eval_results)

            # Generate recommendations for different risk profiles
            for risk_profile in ['Low', 'Medium', 'High']:
                recommendations = generate_portfolio_recommendations(
                    model, stock_dfs, risk_df, feature_columns, risk_profile=risk_profile
                )

                print(f"\n{risk_profile} Risk Portfolio Recommendations:")
                if not recommendations.empty:
                    print(recommendations[['ticker', 'allocation_pct', 'risk_category', 'sharpe_ratio']].head(10))

                    # Save recommendations to CSV
                    recommendations.to_csv(f"{risk_profile.lower()}_risk_portfolio_recommendations.csv", index=False)
                else:
                    print("No recommendations generated")

        except Exception as e:
            print(f"Error in SAC model processing: {str(e)}")
            print("Falling back to traditional risk classification only")
    else:
        print("No valid risk data generated!")

if __name__ == "__main__":
    main()


Using data file: /content/stock_risk_analysis.csv
Original data shape: (1462, 11)
Available columns: ['ticker', 'risk_category', 'risk_score', 'volatility', 'max_drawdown', 'sharpe_ratio', 'avg_daily_return', 'annualized_return', 'avg_volume', 'volume_volatility', 'data_points']
Skipping AAB.TO - insufficient data (1 rows)
Skipping AAUC.TO - insufficient data (1 rows)
Skipping AAV.TO - insufficient data (1 rows)
Skipping ABX.TO - insufficient data (1 rows)
Skipping AC.TO - insufficient data (1 rows)
Skipping ACB.TO - insufficient data (1 rows)
Skipping ACD.TO - insufficient data (1 rows)
Skipping ACQ.TO - insufficient data (1 rows)
Skipping ACX.TO - insufficient data (1 rows)
Skipping ADCO.TO - insufficient data (1 rows)
Skipping ADEN.TO - insufficient data (1 rows)
Skipping ADN.TO - insufficient data (1 rows)
Skipping AEG.TO - insufficient data (1 rows)
Skipping AEM.TO - insufficient data (1 rows)
Skipping AFN.TO - insufficient data (1 rows)
Skipping AG.TO - insufficient data (1 rows)